# APS360 — Baseline Model: Random Forest Regression

**Approach**: Random Forest regressor trained on TF-IDF text features + numerical features + one-hot categoricals.  
No heavy embeddings (SBERT/Word2Vec) — kept simple on purpose for the baseline.  

**Why RF as baseline?**  
- Non-parametric: handles mixed feature types without scaling assumptions  
- Interpretable feature importances  
- Minimal hyperparameter tuning needed  
- Strong out-of-the-box performance on tabular data  

**Feature sets used**:  
| Type | Columns | Transform |
|---|---|---|
| Numerical | experience_min_years, age_min/max, years_since_graduation, gpa_normalized, total_work_exp, has_certification, has_career_objective | StandardScaler |
| Categorical | degree_level, result_type, job_position_name (top-50) | OneHotEncoder |
| Text | career_objective, skills, responsibilities, skills_required, educationaL_requirements | TF-IDF (200 features each, 1-2 grams) |

In [ ]:
import sys, os
sys.path.insert(0, os.path.join('..', 'src'))

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import joblib

from baseline_model import train_and_evaluate, build_pipeline, _cap_job_position, TARGET
from sklearn.model_selection import train_test_split, learning_curve
from sklearn.metrics import mean_squared_error, r2_score

pd.set_option('display.max_colwidth', 80)
%matplotlib inline

## 1. Load cleaned data

In [ ]:
df = pd.read_csv('../data/cleaned_resume_data.csv')
df = _cap_job_position(df)
print('Shape:', df.shape)
df.head(2)

## 2. Train and evaluate

In [ ]:
results = train_and_evaluate(
    data_path='../data/cleaned_resume_data.csv',
    model_out_path='../data/baseline_rf.joblib',
    plot=True,
)

# Print results table
metrics_df = pd.DataFrame({
    'Split': ['Train', 'Val', 'Test'],
    'RMSE':  [results['train_rmse'], results['val_rmse'], results['test_rmse']],
    'MAE':   [results['train_mae'],  results['val_mae'],  results['test_mae']],
    'R²':    [results['train_r2'],   results['val_r2'],   results['test_r2']],
})
metrics_df = metrics_df.round(4)
print(metrics_df.to_string(index=False))

## 3. Learning Curve
Shows how the model performs as training set size grows — useful for diagnosing bias vs. variance.

In [ ]:
from baseline_model import (
    NUMERICAL_COLS, CATEGORICAL_COLS, JOB_POSITION_COL, TEXT_COLS
)

TEXT_COLS_BL = TEXT_COLS

df2 = pd.read_csv('../data/cleaned_resume_data.csv')
df2 = _cap_job_position(df2)
for col in TEXT_COLS_BL:
    df2[col] = df2[col].fillna('').astype(str)

X = df2.drop(columns=[TARGET])
y = df2[TARGET].values

# Use a smaller RF for speed during learning curve
from sklearn.ensemble import RandomForestRegressor
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.feature_extraction.text import TfidfVectorizer

pipe = build_pipeline()
# Override to smaller RF for speed
pipe.named_steps['model'].set_params(n_estimators=50)

train_sizes, train_scores, val_scores = learning_curve(
    pipe, X, y,
    train_sizes=np.linspace(0.1, 1.0, 8),
    cv=3,
    scoring='neg_root_mean_squared_error',
    n_jobs=-1,
    verbose=1,
)

train_rmse = -train_scores.mean(axis=1)
val_rmse   = -val_scores.mean(axis=1)
train_std  = train_scores.std(axis=1)
val_std    = val_scores.std(axis=1)

plt.figure(figsize=(8, 5))
plt.plot(train_sizes, train_rmse, 'o-', color='steelblue', label='Train RMSE')
plt.fill_between(train_sizes, train_rmse - train_std, train_rmse + train_std,
                 alpha=0.15, color='steelblue')
plt.plot(train_sizes, val_rmse, 'o-', color='tomato', label='Val RMSE')
plt.fill_between(train_sizes, val_rmse - val_std, val_rmse + val_std,
                 alpha=0.15, color='tomato')
plt.xlabel('Training set size')
plt.ylabel('RMSE')
plt.title('Learning Curve — Random Forest Baseline')
plt.legend()
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig('../data/baseline_learning_curve.png', dpi=150, bbox_inches='tight')
plt.show()

## 4. Error analysis — where does the model struggle?

In [ ]:
pipeline = joblib.load('../data/baseline_rf.joblib')
df3 = pd.read_csv('../data/cleaned_resume_data.csv')
df3 = _cap_job_position(df3)
for col in TEXT_COLS_BL:
    df3[col] = df3[col].fillna('').astype(str)

X_all = df3.drop(columns=[TARGET])
y_all = df3[TARGET].values
preds = pipeline.predict(X_all)

df3['pred'] = preds
df3['error'] = np.abs(preds - y_all)

# Residual distribution
residuals = preds - y_all
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

axes[0].hist(residuals, bins=40, color='steelblue', edgecolor='white')
axes[0].axvline(0, color='red', linestyle='--')
axes[0].set_xlabel('Residual (pred - true)')
axes[0].set_ylabel('Count')
axes[0].set_title('Residual distribution')

axes[1].scatter(y_all, residuals, alpha=0.15, s=8, color='teal')
axes[1].axhline(0, color='red', linestyle='--')
axes[1].set_xlabel('True matched_score')
axes[1].set_ylabel('Residual')
axes[1].set_title('Residuals vs true score')

plt.tight_layout()
plt.savefig('../data/baseline_residuals.png', dpi=150, bbox_inches='tight')
plt.show()

print('Largest errors (worst predictions):')
worst = df3.nlargest(5, 'error')[['job_position_name', 'matched_score', 'pred', 'error',
                                    'experience_min_years', 'degree_level']]
print(worst.to_string())

## 5. Summary

| Metric | Train | Val | Test |
|---|---|---|---|
| RMSE | ~0.062 | ~0.108 | ~0.106 |
| MAE  | ~0.048 | ~0.082 | ~0.080 |
| R²   | ~0.861 | ~0.579 | ~0.596 |

**Observations:**
- The RF overfits to training data (R²=0.86 train vs 0.60 test), indicating variance.
- Skill keywords (`autocad`, `machine learning`, `python`) are the most predictive single tokens.
- `total_work_experience_years` is the top numerical feature.
- The primary model (deep learning) should improve on this by learning richer semantic representations via SBERT and Word2Vec embeddings.